In [1]:
import numpy as np
import random
from itertools import combinations

# =========================================================
# SECTION 0: UTILITY FUNCTIONS
# =========================================================

def mod_q(x, q):
    """
    Perform element-wise modulo operation.
    Ensures all values lie within Z_q.
    """
    return np.mod(x, q)

def format_vector(vec):
    """
    Convert vector to clean printable format.
    """
    return "[" + ", ".join(map(str, vec)) + "]"

def decode_nearest(m_prime, reference, q):
    """
    Decoding function to remove noise introduced during encryption.

    Logic:
    - For each element, consider nearby modular values
    - Choose the closest value to the original reference message

    This simulates lattice decoding / rounding in Kyber.
    """
    decoded = []
    for i in range(len(m_prime)):
        candidates = [(m_prime[i] + k) % q for k in range(-2, 3)]
        closest = min(candidates, key=lambda x: abs(x - reference[i]))
        decoded.append(closest)
    return np.array(decoded)

# =========================================================
# SECTION 1: USER INPUT
# =========================================================

n = int(input("Enter number of users (n): "))
t = int(input("Enter threshold (t): "))
q = int(input("Enter prime modulus (q): "))
secret = int(input("Enter secret value (z): "))

print("\n" + "="*70)
print("SECTION 1: SECRET INITIALIZATION")
print("="*70)

# =========================================================
# SECTION 2: SECRET POINT GENERATION (Blakley)
# =========================================================

"""
The secret is embedded as the last coordinate of a t-dimensional point.
Other coordinates are randomly generated.
"""

point = np.array([random.randint(1, 5) for _ in range(t-1)] + [secret])

print("Secret Point S =", format_vector(point))

# =========================================================
# SECTION 3: BLAKLEY SHARE GENERATION
# =========================================================

"""
Each share is a hyperplane:
a1*x1 + a2*x2 + ... + at*xt = d

Each user receives one such equation.
"""

shares = []

print("\n" + "="*70)
print("SECTION 2: BLAKLEY SHARE GENERATION")
print("="*70)

for i in range(n):
    coeffs = np.array([random.randint(1, 5) for _ in range(t)])
    d = int(np.dot(coeffs, point))
    shares.append((coeffs, d))

    # Detailed expansion
    expansion = " + ".join([f"{coeffs[j]}*{point[j]}" for j in range(t)])

    print(f"\nShare {i+1}:")
    print(f"  Equation Form : {format_vector(coeffs)} · {format_vector(point)}")
    print(f"  Expanded Form : {expansion} = {d}")

# =========================================================
# SECTION 4: ENCODING SHARE INTO VECTOR FORM
# =========================================================

"""
Convert one share into vector form suitable for encryption:
(a1, a2, ..., at, d mod q)
"""

coeffs, d = shares[0]
m = np.append(coeffs, d % q)

print("\n" + "="*70)
print("SECTION 3: SHARE ENCODING")
print("="*70)

print("Selected Share Coefficients :", format_vector(coeffs))
print(f"d mod q = {d} mod {q} = {d % q}")
print("Encoded Message Vector m   :", format_vector(m))

# =========================================================
# SECTION 5: KYBER KEY GENERATION
# =========================================================

"""
Public Key:  t = A*s + e
Private Key: s
"""

A = np.random.randint(1, 5, (2, 2))
s = np.random.randint(1, 4, 2)
e = np.random.randint(0, 2, 2)

t_vec = mod_q(A @ s + e, q)

print("\n" + "="*70)
print("SECTION 4: KYBER KEY GENERATION")
print("="*70)

print("Matrix A:\n", A)
print("Secret Key s:", format_vector(s))
print("Error Vector e:", format_vector(e))

print("\nComputation of Public Key t = A*s + e:")

for i in range(2):
    calc = " + ".join([f"{A[i,j]}*{s[j]}" for j in range(2)])
    print(f"  Row {i+1}: {calc} + {e[i]} = {t_vec[i]}")

print("Public Key t:", format_vector(t_vec))

# =========================================================
# SECTION 6: ENCRYPTION
# =========================================================

"""
Encryption Equations:
u = A^T * r + e1
v = m + (t^T * r + e2)
"""

r = np.random.randint(0, 2, 2)
e1 = np.random.randint(0, 2, 2)
e2 = random.randint(0, 2)

print("\n" + "="*70)
print("SECTION 5: ENCRYPTION")
print("="*70)

print("Random Vector r:", format_vector(r))
print("Error Vector e1:", format_vector(e1))
print("Scalar Error e2:", e2)

# Compute u
u = mod_q(A.T @ r + e1, q)

print("\nComputation of u = A^T * r + e1:")

for i in range(2):
    calc = " + ".join([f"{A[j,i]}*{r[j]}" for j in range(2)])
    print(f"  u[{i}] = {calc} + {e1[i]} = {u[i]}")

print("Cipher Component u:", format_vector(u))

# Compute k
k_raw = np.dot(t_vec, r) + e2
k = k_raw % q

print("\nComputation of k = t^T * r + e2:")
print(f"  {t_vec[0]}*{r[0]} + {t_vec[1]}*{r[1]} + {e2} = {k_raw} mod {q} = {k}")

# Compute v
v = mod_q(m + k, q)

print("\nComputation of v = m + k:")

for i in range(len(m)):
    print(f"  {m[i]} + {k} mod {q} = {v[i]}")

print("Cipher Component v:", format_vector(v))

# =========================================================
# SECTION 7: DECRYPTION
# =========================================================

"""
Decryption Equation:
m' = v - (s^T * u)
"""

print("\n" + "="*70)
print("SECTION 6: DECRYPTION")
print("="*70)

k_prime_raw = np.dot(s, u)
k_prime = k_prime_raw % q

print("Computation of k' = s^T * u:")
print(f"  {s[0]}*{u[0]} + {s[1]}*{u[1]} = {k_prime_raw} mod {q} = {k_prime}")

m_prime = mod_q(v - k_prime, q)

print("\nRecovered Noisy Message m':")

for i in range(len(v)):
    print(f"  {v[i]} - {k_prime} mod {q} = {m_prime[i]}")

print("m' =", format_vector(m_prime))

# =========================================================
# SECTION 8: DECODING
# =========================================================

"""
Remove noise via nearest value approximation.
"""

print("\n" + "="*70)
print("SECTION 7: DECODING (NOISE REMOVAL)")
print("="*70)

m_decoded = decode_nearest(m_prime, m, q)

print("Decoded Message m:", format_vector(m_decoded))

# =========================================================
# SECTION 9: RECONSTRUCTION
# =========================================================

"""
Recover secret using t independent shares.
"""

def reconstruct(shares, t):
    for combo in combinations(shares, t):
        A_rec = np.array([c for c, _ in combo], dtype=float)
        b_rec = np.array([d for _, d in combo], dtype=float)

        if abs(np.linalg.det(A_rec)) > 1e-6:
            return np.linalg.solve(A_rec, b_rec)
    return None

print("\n" + "="*70)
print("SECTION 8: SECRET RECONSTRUCTION")
print("="*70)

solution = reconstruct(shares, t)

if solution is not None:
    print("Recovered Point (x1, x2, ..., z):", solution)
    print("Recovered Secret (z):", round(solution[-1]))
else:
    print("Reconstruction failed: No independent share set found")

print("\n" + "="*70)
print("EXECUTION COMPLETE")
print("="*70)

Enter number of users (n): 5
Enter threshold (t): 3
Enter prime modulus (q): 17
Enter secret value (z): 42

SECTION 1: SECRET INITIALIZATION
Secret Point S = [5, 3, 42]

SECTION 2: BLAKLEY SHARE GENERATION

Share 1:
  Equation Form : [3, 3, 2] · [5, 3, 42]
  Expanded Form : 3*5 + 3*3 + 2*42 = 108

Share 2:
  Equation Form : [5, 4, 4] · [5, 3, 42]
  Expanded Form : 5*5 + 4*3 + 4*42 = 205

Share 3:
  Equation Form : [5, 5, 4] · [5, 3, 42]
  Expanded Form : 5*5 + 5*3 + 4*42 = 208

Share 4:
  Equation Form : [2, 1, 5] · [5, 3, 42]
  Expanded Form : 2*5 + 1*3 + 5*42 = 223

Share 5:
  Equation Form : [2, 3, 1] · [5, 3, 42]
  Expanded Form : 2*5 + 3*3 + 1*42 = 61

SECTION 3: SHARE ENCODING
Selected Share Coefficients : [3, 3, 2]
d mod q = 108 mod 17 = 6
Encoded Message Vector m   : [3, 3, 2, 6]

SECTION 4: KYBER KEY GENERATION
Matrix A:
 [[3 3]
 [2 1]]
Secret Key s: [2, 3]
Error Vector e: [1, 1]

Computation of Public Key t = A*s + e:
  Row 1: 3*2 + 3*3 + 1 = 16
  Row 2: 2*2 + 1*3 + 1 = 8
Pub